In [ ]:
nseg_len = 60 * 144          # 8640 points per segment
step     = nseg_len // 2     # 4320 points (50% overlap)

for r in recs:
    N = r['WSPD_i'].size
    nseg = (N - nseg_len) // step + 1
    starts = np.arange(nseg) * step
    r['segs'] = {v: np.array([r[v + '_i'][s:s + nseg_len] for s in starts])
                 for v in ('WSPD', 'UWND', 'VWND')}

print(recs[0]['segs']['WSPD'].shape)     # should be (10, 8640)

In [ ]:
from scipy.signal import detrend

#Detrended data
segs = recs[0]['segs']['WSPD']          # shape (10, 8640), 2015 record
L = segs.shape[1]                       # 8640 Samples per day in 2015 
dt_days = 10.0 / 1440.0                 #10 min in days
f = np.fft.fftfreq(L, d=dt_days)        #cpd = 0.0167
pos = f >= 0
fp = f[pos]

def avg_energy(x, w=None):
    """Mean |c|^2 over segments, with c = fft(x*w)/N like your pressure code."""
    if w is None:
        w = np.ones(L)
    c = np.fft.fft(x * w, axis=1) / L
    S = (np.abs(c)**2).mean(axis=0)
    S = S / np.mean(w**2)               # window power correction so levels compare
    return S[pos]

#Hanning Window Applied
hann = np.hanning(L)
x_det = detrend(segs, axis=1, type='linear')

S_raw  = avg_energy(segs)
S_det  = avg_energy(x_det)
S_hann = avg_energy(x_det, hann)

#Plots figure 
fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(fp[1:], S_raw[1:],  label='raw')
ax.loglog(fp[1:], S_det[1:],  label='detrended')
ax.loglog(fp[1:], S_hann[1:], label='detrended + Hanning')
ax.set_xlabel('frequency (cpd)', fontsize=14)
ax.set_ylabel('|c|$^2$ ((m s$^{-1}$)$^2$)', fontsize=14)
ax.set_title('WSPD spectral energy, 2015 record (10 segments averaged)', fontsize=14)
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()